# L4 — Orderbook Dynamics

Charts O1–O5.

**Memory strategy**: BBO-only first; full OB processed per exchange (~4 GB peak vs ~25 GB all-at-once).

In [ ]:
import os, gc, numpy as np, pandas as pd

DATA_DIR    = os.environ.get("DATA_DIR",    "/data/parquet")
REPORTS_DIR = os.environ.get("REPORTS_DIR", "/data/reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

from mda.loader import load_orderbook, load_orderbook_updates
from mda.layer4.bbo          import compute_bbo, spread_stats
from mda.layer4.update_rates import compute_update_rate_by_depth, compute_delta_compression_ratio
from mda.layer4.levels        import compute_level_lifetimes_polars, level_lifetime_stats
from mda.plots.charts import (
    plot_update_rate_by_depth, plot_bbo_spread_depth,
    plot_delta_compression_ts, plot_book_activity_heatmap,
    plot_level_lifetime_hist, save_figure,
)
print("imports OK")

## O2 — BBO Spread & Depth (level==0 only)

In [ ]:
ob_bbo = load_orderbook(DATA_DIR, bbo_only=True)
EXCHANGES = ob_bbo["exchange"].unique().tolist()
print(f"BBO rows: {len(ob_bbo):,} | exchanges: {EXCHANGES}")

In [ ]:
bbo = compute_bbo(ob_bbo)
s   = spread_stats(bbo)
print("BBO spread stats (bps):")
print(s.to_string(index=False))

In [ ]:
bbo["ts_dt"] = pd.to_datetime(bbo["received_at"], utc=True, format="mixed")
fig = plot_bbo_spread_depth(bbo)
fig.show()
save_figure(fig, "O2_bbo_spread_depth", REPORTS_DIR)
del ob_bbo, bbo, s
gc.collect()
print("BBO memory freed")

## O1/O4 — Update Rate by Depth & Book Activity (per exchange)

In [ ]:
update_rate_parts = []
for exch in EXCHANGES:
    print(f"Loading {exch}...")
    ob_exch = load_orderbook(DATA_DIR, exchanges=[exch])
    print(f"  {exch}: {len(ob_exch):,} rows")
    update_rate_parts.append(compute_update_rate_by_depth(ob_exch))
    del ob_exch
    gc.collect()

update_rate = pd.concat(update_rate_parts, ignore_index=True)
del update_rate_parts
gc.collect()
print("Update rate computed")
print(update_rate[update_rate["level"] < 10].to_string(index=False))

In [ ]:
fig = plot_update_rate_by_depth(update_rate[update_rate["level"] < 20])
fig.show()
save_figure(fig, "O1_update_rate_by_depth", REPORTS_DIR)

In [ ]:
for exch in EXCHANGES:
    fig = plot_book_activity_heatmap(update_rate, exchange=exch)
    fig.show()
    save_figure(fig, f"O4_book_activity_{exch}", REPORTS_DIR)
del update_rate
gc.collect()
print("update_rate freed")

## O3 — Delta Compression Ratio (per exchange)

In [ ]:
ob_updates = load_orderbook_updates(DATA_DIR)
print(f"OB updates: {len(ob_updates):,} rows")

delta_parts = []
for exch in EXCHANGES:
    print(f"  {exch}...")
    ob_exch = load_orderbook(DATA_DIR, exchanges=[exch])
    ob_upd_exch = ob_updates[ob_updates["exchange"] == exch]
    try:
        delta_parts.append(compute_delta_compression_ratio(ob_exch, ob_upd_exch))
    except Exception as e:
        print(f"  {exch}: skipped ({e})")
    del ob_exch
    gc.collect()

del ob_updates
gc.collect()

if delta_parts:
    delta = pd.concat(delta_parts, ignore_index=True)
    del delta_parts
    fig = plot_delta_compression_ts(delta)
    fig.show()
    save_figure(fig, "O3_delta_compression_ts", REPORTS_DIR)

## O5 — Level Lifetimes (Polars lazy scan)

In [ ]:
lifetime_parts = []
for exch in EXCHANGES:
    try:
        lt = compute_level_lifetimes_polars(DATA_DIR, exchange=exch, symbol="BTCUSDT")
        if lt is not None and len(lt) > 0:
            lifetime_parts.append(lt)
            print(f"{exch}: {len(lt):,} levels")
    except Exception as e:
        print(f"{exch}: skipped ({e})")

if lifetime_parts:
    lifetimes = pd.concat(lifetime_parts, ignore_index=True)
    print(level_lifetime_stats(lifetimes).to_string(index=False))
    fig = plot_level_lifetime_hist(lifetimes)
    fig.show()
    save_figure(fig, "O5_level_lifetime_hist", REPORTS_DIR)
else:
    print("No level lifetime data")